# Estudio 06 · Fase 4 · Señal regional de El Niño (Niño 1+2)

Preregistro `PHASE4` (commit `fb403ca`, aprobado por Rosa con la verificación del archivo crudo). Base: L + 5a.

- **4a:** C3 + anomalía Niño 1+2 del último mes disponible al origen.
- **4b:** C3 + media de los 3 últimos meses disponibles al origen.

Un mes se usa solo desde su `fecha_disponible` (día 10 del mes siguiente) si es igual o anterior a `origen_cierre`.
Regla de 5 temporadas de las fases 2 y 3; se adopta con «gana» en >= 2 de 3 horizontes y sin «pierde» ni «compensa».


In [1]:
import json
import sys

import numpy as np
import pandas as pd

from denguekay.config.paths import RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.model import tracking
from denguekay.model.device import xgboost_settings

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase2_calibration as p2
import phase3_onsets as p3
import phase4_enso as p4
import phase5_counts as p5
import phase6_other_families as p6
import preregistration as pr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
METRICS = STUDY / "docs" / "metricas" / "fase4.json"
device = xgboost_settings()
inp = p3.phase3_inputs()
enso = p4.read_enso()
origins = p4.enso_at_origins(inp["origenes"], enso)
device


{'device': 'cuda', 'n_jobs': 12}

## 1 · Guardas G21, G23 y G24 (G22 es el test de no fuga)


In [2]:
g21 = p4.guard_g21(origins[origins.nino12_anom_ultimo.notna()])
g23 = p4.guard_g23(origins)
base = p6.l_plus_5a(inp["serie"])
saved5 = json.loads((STUDY / "docs" / "metricas" / "fase5.json").read_text(encoding="utf-8"))
g24 = p6.guard_g16_reproduction(base["evaluaciones"], saved5["veredictos"]["5a"]["metricas"])
if not (g21["pasa"] and g23["pasa"] and g24["pasa"]):
    raise RuntimeError(f"Falla una guarda: se detiene la fase. G21={g21} G23={g23} G24={g24}")
{"G21": g21, "G23": g23, "G24": g24}


{'G21': {'origenes': 71175, 'con_mes_posterior': 0, 'pasa': True},
 'G23': {'origenes_desde_2005': 71110, 'sin_valor': 0, 'pasa': True},
 'G24': {'diferencia_maxima': 4.9451664985156185e-11,
  'filas': 18,
  'pasa': True}}

In [3]:
# Rango de la variable en los orígenes desde 2005 (lectura)
since = pd.to_datetime(origins.origen_cierre).ge(pd.Timestamp("2005-01-01"))
rango = origins.loc[since, list(p4.COLUMNS)].agg(["min", "max", "mean"]).round(2)
rango


,nino12_anom_ultimo,nino12_anom_media3
min,-1.90,-1.74
max,3.30,3.11
mean,0.02,0.03


## 2 · Variantes (5 semillas)


In [4]:
variants = {v: p4.run_variant(inp, origins, v, pr.SEEDS) for v in pr.PHASE4_COLUMNS}
p4.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for v, r in variants.items():
    pd.concat(r["predicciones"].values(), ignore_index=True).to_csv(p4.OUTPUT_DIR / f"predicciones_{v}.csv.gz", index=False)
    r["ajustes"].drop(columns=["corte"]).to_csv(p4.OUTPUT_DIR / f"ajustes_{v}.csv", index=False)


## 3 · Veredictos preregistrados


In [5]:
dec = {v: p3.compare(r, base) for v, r in variants.items()}
adopted = [v for v in dec if pr.adopt_phase4(dec[v]["veredicto"])]
print("Adoptadas:", adopted)
pd.concat({v: d["veredicto"] for v, d in dec.items()}, names=["variante"]).reset_index(level=0)


Adoptadas: []


,variante,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,4a,2,empate,False,False,None,NaN,0,1,2,2,1,1,1,1,0,1
1,4a,3,empate,False,False,None,razon_mae,0,2,2,3,2,1,1,1,0,0
2,4a,4,empate,False,False,None,razon_mae,1,2,2,4,2,1,2,1,0,1
0,4b,2,empate,False,False,None,NaN,0,1,2,1,1,0,0,1,0,1
1,4b,3,empate,False,False,None,brier,0,2,3,2,1,1,0,1,0,2
2,4b,4,empate,False,False,None,"auprc, brier",1,3,3,2,2,1,0,1,0,2


In [6]:
pd.concat({v: d["antelacion"] for v, d in dec.items()}, names=["variante"]).round(3)


horizonte  episodios  diferencia  ic95_inferior  ic95_superior  diferencia_temporada_2010  diferencia_temporada_2015  diferencia_temporada_2016  diferencia_temporada_2022  diferencia_temporada_2023  \
variante                                                                                                                                                                                                            
4a       0          2        193      -0.021         -0.047          0.005                      0.000                     -0.029                      0.000                     -0.079                      0.000   
         1          3        193      -0.005         -0.031          0.021                      0.000                      0.000                      0.000                     -0.026                      0.000   
         2          4        193      -0.016         -0.036          0.000                      0.000                      0.000                      0.000                     -0.053                     -0.018   
4b       0          2        193      -0.021         -0.052          0.005                      0.033                      0.000                     -0.029                     -0.079                     -0.018   
         1          3        193      -0.026         -0.062          0.005                      0.000                      0.029                     -0.059                     -0.079                     -0.018   
         2          4        193      -0.026         -0.052          0.000                      0.000                      0.000                      0.000                     -0.079                     -0.035   

            episodios_temporada_2010  episodios_temporada_2015  episodios_temporada_2016  episodios_temporada_2022  episodios_temporada_2023  
variante                                                                                                                                      
4a       0                        30                        34                        34                        38                        57  
         1                        30                        34                        34                        38                        57  
         2                        30                        34                        34                        38                        57  
4b       0                        30                        34                        34                        38                        57  
         1                        30                        34                        34                        38                        57  
         2                        30                        34                        34                        38                        57

In [7]:
cols = ["horizonte", "bloque", "positivos_cand", "f1_050_cand", "f1_050_ref", "auprc_cand", "auprc_ref", "brier_cand",
        "brier_ref", "razon_mae_cand", "razon_mae_ref"]
pd.concat({v: d["metricas"][cols] for v, d in dec.items()}, names=["variante"]).round(3)


horizonte          bloque  positivos_cand  f1_050_cand  f1_050_ref  auprc_cand  auprc_ref  brier_cand  brier_ref  razon_mae_cand  razon_mae_ref
variante                                                                                                                                                    
4a       3           2  temporada_2010           234.0        0.712       0.722       0.780      0.780       0.030      0.030           0.890          0.907
         8           2  temporada_2015           557.0        0.765       0.772       0.882      0.885       0.052      0.051           0.903          0.889
         9           2  temporada_2016           395.0        0.660       0.657       0.757      0.757       0.059      0.059           1.009          1.030
         15          2  temporada_2022           431.0        0.694       0.712       0.782      0.781       0.053      0.054           0.917          0.946
         16          2  temporada_2023          1039.0        0.857       0.857       0.948      0.947       0.061      0.062           0.803          0.808
         17          2  temporada_2024           857.0        0.801       0.799       0.896      0.893       0.073      0.074           0.999          1.136
         21          3  temporada_2010           234.0        0.622       0.631       0.706      0.706       0.036      0.036           0.872          0.891
         26          3  temporada_2015           557.0        0.631       0.649       0.834      0.839       0.065      0.064           0.846          0.828
         27          3  temporada_2016           395.0        0.627       0.627       0.710      0.709       0.065      0.065           1.012          1.042
         33          3  temporada_2022           431.0        0.691       0.704       0.774      0.771       0.055      0.056           0.875          0.927
         34          3  temporada_2023          1039.0        0.837       0.835       0.932      0.930       0.072      0.073           0.752          0.763
         35          3  temporada_2024           857.0        0.790       0.788       0.882      0.876       0.078      0.079           1.057          1.238
         39          4  temporada_2010           234.0        0.436       0.476       0.600      0.600       0.042      0.042           0.851          0.876
         44          4  temporada_2015           557.0        0.459       0.480       0.802      0.811       0.076      0.075           0.814          0.793
         45          4  temporada_2016           395.0        0.594       0.590       0.658      0.660       0.070      0.070           1.026          1.062
         51          4  temporada_2022           431.0        0.632       0.652       0.746      0.743       0.059      0.060           0.858          0.929
         52          4  temporada_2023          1039.0        0.812       0.811       0.923      0.922       0.080      0.081           0.720          0.733
         53          4  temporada_2024           857.0        0.765       0.761       0.849      0.840       0.087      0.089           1.130          1.362
4b       3           2  temporada_2010           234.0        0.724       0.722       0.786      0.780       0.030      0.030           0.903          0.907
         8           2  temporada_2015           557.0        0.770       0.772       0.884      0.885       0.051      0.051           0.888          0.889
         9           2  temporada_2016           395.0        0.658       0.657       0.757      0.757       0.058      0.059           1.020          1.030
         15          2  temporada_2022           431.0        0.696       0.712       0.782      0.781       0.053      0.054           0.901          0.946
         16          2  temporada_2023          1039.0        0.858       0.857       0.948      0.947       0.062      0.062           0.813          0.808
         17          2  temporada_2024           857.0        0.797       0.799       0.

In [8]:
crosses = []
if len(adopted) > 1:
    c = p3.compare(variants["4b"], variants["4a"])
    wins = pr.adopt_phase4(c["veredicto"])
    crosses.append({"candidata": "4b", "frente_a": "4a", "adopta": wins,
                    "veredicto": "; ".join(f"h{r.horizonte}: {r.veredicto}" for r in c["veredicto"].itertuples())})
    final = "4b" if wins else "4a"
else:
    final = adopted[0] if adopted else "L + 5a"
print("Base tras la fase 4:", final, crosses)


Base tras la fase 4: L + 5a []


## 4 · Lecturas que no deciden


In [9]:
pd.concat({v: d["lectura_sin_2010"][["horizonte", "veredicto", "vetos"]] for v, d in dec.items()},
          names=["variante"]).reset_index(level=0)


,variante,horizonte,veredicto,vetos
0,4a,2,empate,None
1,4a,3,empate,None
2,4a,4,empate,None
0,4b,2,empate,None
1,4b,3,empate,None
2,4b,4,empate,None


In [10]:
mae_rule = {v: p5.compare(r, base) for v, r in variants.items()}
pd.concat({v: d["veredicto"][["horizonte", "veredicto", "vetos"]] for v, d in mae_rule.items()},
          names=["variante"]).reset_index(level=0)


,variante,horizonte,veredicto,vetos
0,4a,2,empate,None
1,4a,3,empate,None
2,4a,4,empate,None
0,4b,2,empate,None
1,4b,3,empate,None
2,4b,4,empate,None


In [11]:
rd = {k: p2.readings(v, inp["serie"]) for k, v in {"L + 5a": base, **variants}.items()}
recall = pd.concat({k: v["recall_por_tipo"] for k, v in rd.items()}, names=["variante"]).reset_index(level=0)
recall


,variante,horizonte,bloque,tipo,semanas,recall,semanas_rango,recall_rango
4,L + 5a,2,temporada_2010,brote_nuevo,51.0,0.117647,0,0.058824
5,L + 5a,2,temporada_2010,en_curso,183.0,0.863388,0,0.032787
14,L + 5a,2,temporada_2015,brote_nuevo,60.0,0.066667,0,0.000000
15,L + 5a,2,temporada_2015,en_curso,497.0,0.770624,0,0.014085
16,L + 5a,2,temporada_2016,brote_nuevo,53.0,0.132075,0,0.000000
17,L + 5a,2,temporada_2016,en_curso,342.0,0.804094,0,0.002924
18,L + 5a,2,temporada_2017,brote_nuevo,79.0,0.329114,0,0.000000
19,L + 5a,2,temporada_2017,en_curso,460.0,0.815217,0,0.004348
26,L + 5a,2,temporada_2022,brote_nuevo,49.0,0.122449,0,0.040816
27,L + 5a,2,temporada_2022,en_curso,382.0,0.769634,0,0.005236


## 5 · Salidas: métricas y MLflow


In [12]:
to_json = lambda d: {k: diag.to_json(v) for k, v in d.items() if isinstance(v, pd.DataFrame)}
content = {
    "dispositivo": device, "guardas": {"G21": g21, "G23": g23, "G24": g24},
    "rango_desde_2005": rango.reset_index().to_dict(orient="records"),
    "veredictos": {v: to_json(d) for v, d in dec.items()}, "adoptadas": adopted, "cruces": crosses, "final": final,
    "lectura_regla_mae": {v: to_json(d) for v, d in mae_rule.items()},
    "recall_por_tipo": diag.to_json(recall),
    "hu0007_4": {k: diag.to_json(v["hu0007_4"]) for k, v in rd.items()},
}
content = json.loads(json.dumps(content, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
verdict_text = lambda d: "; ".join(f"h{r.horizonte}: {r.veredicto}" for r in d["veredicto"].itertuples())
tracking.log_study(
    "mejora_modelo_fase4_nino", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "4", "preregistro": "fb403ca", "final": final,
                       **{f"veredicto.{v}": verdict_text(d) for v, d in dec.items()},
                       **{f"xgb.{k}": str(x) for k, x in device.items()}},
            "metricas": {}},
    artifacts=[METRICS], tables={f"veredicto_{v}": d["veredicto"] for v, d in dec.items()},
    description="Fase 4 del estudio 06: anomalía Niño 1+2 (último mes y media de 3 meses) sobre L + 5a.")


C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: estudio 01c630b438a6469da4e5909bb6b6262d en 'denguekay/mejora_modelo_fase4_nino' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'01c630b438a6469da4e5909bb6b6262d'